# 1. Objective of Final Pipeline

The objective of this notebook is to prepare the final fraud detection pipeline for deployment.

After selecting and evaluating the final machine learning model, this notebook focuses on building an end-to-end prediction workflow that can be reused in the deployment stage.

The pipeline should be able to:

- accept transaction input data
- apply the same preprocessing used during training
- generate a fraud prediction
- return the fraud probability
- support integration with FastAPI and Streamlit

This notebook also exports the final model and preprocessing objects, saves an example input/output schema, and documents the handover requirements for deployment.

The final output of this notebook will serve as the technical bridge between machine learning development and application deployment.

# 2. Import Libraries

In this section, the required Python libraries for loading model objects, building the prediction pipeline, testing sample predictions, and exporting deployment files are imported.

In [1]:
import pandas as pd
import numpy as np
import os
import joblib
import json
from pathlib import Path

# 3. Load Final Model Components

The final model and preprocessing objects are loaded in this section.

Depending on the final model selected by the project, this may include:

- the trained model
- the fitted scaler
- the selected decision threshold
- the feature column order

These components are necessary to ensure that future predictions use the same structure and preprocessing steps as the training pipeline.

In [2]:
final_model = joblib.load("../models/final/best_xgboost_model.pkl")
print("Final model loaded successfully.")

Final model loaded successfully.


In [3]:
scaler = joblib.load("../models/scaler.pkl")
print("Scaler loaded successfully.")

Scaler loaded successfully.


In [4]:
X_train_scaled = pd.read_csv("../data/processed/X_train_scaled.csv")
feature_columns = X_train_scaled.columns.tolist()

print("Number of features:", len(feature_columns))
print("First 5 features:", feature_columns[:5])

Number of features: 30
First 5 features: ['Time', 'V1', 'V2', 'V3', 'V4']


In [5]:
selected_threshold = 0.5
print("Selected threshold:", selected_threshold)

Selected threshold: 0.5


# 4. Build End-to-End Prediction Pipeline

In this section, an end-to-end prediction function is created.

The function should:

1. receive raw transaction input
2. convert the input into a DataFrame
3. ensure the feature order matches training
4. apply scaling if required
5. generate fraud probability
6. convert probability into a final fraud prediction based on the selected threshold

This function will later be reused inside the FastAPI backend.

In [6]:
def predict_transaction(input_data, model, scaler, feature_columns, threshold=0.5):
    """
    Predict whether a transaction is fraudulent.

    Parameters:
        input_data (dict): dictionary of feature values
        model: trained machine learning model
        scaler: fitted scaler object
        feature_columns (list): ordered feature names
        threshold (float): probability threshold for fraud classification

    Returns:
        dict: prediction result with class and probability
    """
    
    # Convert input into DataFrame
    input_df = pd.DataFrame([input_data])

    # Ensure correct column order
    input_df = input_df[feature_columns]

    # Apply scaling
    input_scaled = scaler.transform(input_df)

    # Convert back to DataFrame for consistency
    input_scaled_df = pd.DataFrame(input_scaled, columns=feature_columns)

    # Predict probability
    fraud_probability = model.predict_proba(input_scaled_df)[:, 1][0]

    # Apply threshold
    predicted_class = int(fraud_probability >= threshold)

    return {
        "prediction": predicted_class,
        "fraud_probability": float(fraud_probability),
        "threshold": threshold,
        "label": "Fraud" if predicted_class == 1 else "Not Fraud"
    }

# 5. Test the Pipeline with Sample Input

The pipeline is tested with a sample transaction input to confirm that it works correctly from end to end.

This test helps verify:

- correct feature ordering
- successful preprocessing
- successful prediction output
- deployment readiness

In [7]:
X_test_raw = pd.read_csv("../data/processed/X_test_raw.csv")
sample_input = X_test_raw.iloc[0].to_dict()
sample_input

{'Time': 160760.0,
 'V1': -0.674466064578314,
 'V2': 1.40810501967799,
 'V3': -1.11062205357093,
 'V4': -1.32836577843066,
 'V5': 1.38899603254837,
 'V6': -1.30843906707795,
 'V7': 1.88587890268717,
 'V8': -0.614232966299775,
 'V9': 0.311652212453101,
 'V10': 0.65075700363522,
 'V11': -0.857784661547805,
 'V12': -0.229961445775592,
 'V13': -0.19981700479103,
 'V14': 0.266371326329879,
 'V15': -0.0465441684754424,
 'V16': -0.741398089749789,
 'V17': -0.605616644106022,
 'V18': -0.39256818789208,
 'V19': -0.162648311024695,
 'V20': 0.394321820843914,
 'V21': 0.0800842396026648,
 'V22': 0.810033595602455,
 'V23': -0.224327230436412,
 'V24': 0.707899237446867,
 'V25': -0.13583702273753,
 'V26': 0.0451021964988772,
 'V27': 0.533837219064273,
 'V28': 0.291319252625364,
 'Amount': 23.0}

In [8]:
sample_result = predict_transaction(
    input_data=sample_input,
    model=final_model,
    scaler=scaler,
    feature_columns=feature_columns,
    threshold=selected_threshold
)

sample_result

{'prediction': 0,
 'fraud_probability': 2.588052723240253e-07,
 'threshold': 0.5,
 'label': 'Not Fraud'}

In [9]:
X_test_raw = pd.read_csv("../data/processed/X_test_raw.csv")
sample_input = X_test_raw.iloc[0].to_dict()
sample_input

{'Time': 160760.0,
 'V1': -0.674466064578314,
 'V2': 1.40810501967799,
 'V3': -1.11062205357093,
 'V4': -1.32836577843066,
 'V5': 1.38899603254837,
 'V6': -1.30843906707795,
 'V7': 1.88587890268717,
 'V8': -0.614232966299775,
 'V9': 0.311652212453101,
 'V10': 0.65075700363522,
 'V11': -0.857784661547805,
 'V12': -0.229961445775592,
 'V13': -0.19981700479103,
 'V14': 0.266371326329879,
 'V15': -0.0465441684754424,
 'V16': -0.741398089749789,
 'V17': -0.605616644106022,
 'V18': -0.39256818789208,
 'V19': -0.162648311024695,
 'V20': 0.394321820843914,
 'V21': 0.0800842396026648,
 'V22': 0.810033595602455,
 'V23': -0.224327230436412,
 'V24': 0.707899237446867,
 'V25': -0.13583702273753,
 'V26': 0.0451021964988772,
 'V27': 0.533837219064273,
 'V28': 0.291319252625364,
 'Amount': 23.0}

# 6. Export Model and Preprocessing Objects

The final deployment objects are exported in this section so they can be loaded by the backend application.

These exported files may include:

- final trained model
- fitted scaler
- selected threshold
- feature column order

In [10]:
os.makedirs("../deployment_artifacts", exist_ok=True)

joblib.dump(final_model, "../deployment_artifacts/final_model.pkl")
joblib.dump(scaler, "../deployment_artifacts/scaler.pkl")
joblib.dump(feature_columns, "../deployment_artifacts/feature_columns.pkl")
joblib.dump(selected_threshold, "../deployment_artifacts/threshold.pkl")

print("Deployment artifacts exported successfully.")

Deployment artifacts exported successfully.


# 7. Save Example Input/Output Schema

An example input and output schema is saved to help the backend and frontend teams understand the required request and response structure.

This is useful for API integration and testing.

example_input_schema = {
    feature: 0.0 for feature in feature_columns
}

In [11]:
example_input_schema = {
    feature: 0.0 for feature in feature_columns
}

In [12]:
example_output_schema = {
    "prediction": 0,
    "fraud_probability": 0.1234,
    "threshold": selected_threshold,
    "label": "Not Fraud"
}

In [13]:
with open("../deployment_artifacts/example_input_schema.json", "w") as f:
    json.dump(example_input_schema, f, indent=4)

with open("../deployment_artifacts/example_output_schema.json", "w") as f:
    json.dump(example_output_schema, f, indent=4)

print("Example input/output schemas saved successful+ly.")

Example input/output schemas saved successful+ly.


In [14]:
with open("../deployment_artifacts/sample_request.json", "w") as f:
    json.dump(sample_input, f, indent=4)

with open("../deployment_artifacts/sample_response.json", "w") as f:
    json.dump(sample_result, f, indent=4)

print("Sample request/response saved successfully.")

Sample request/response saved successfully.


# 8. Deployment Notes for FastAPI

The FastAPI backend will be responsible for loading the exported deployment artifacts and serving fraud predictions through an API endpoint.

## Recommended FastAPI responsibilities

1. Load:
   - `final_model.pkl`
   - `scaler.pkl`
   - `feature_columns.pkl`
   - `threshold.pkl`

2. Create a `/predict` POST endpoint

3. Accept transaction data as JSON input

4. Pass the input into the prediction pipeline

5. Return:
   - prediction
   - fraud probability
   - threshold
   - label

## Recommended API flow

Client Input → FastAPI Endpoint → Prediction Function → Model Output → JSON Response

## Important notes

- The API must validate that all required features are present
- The feature order must always match training order
- Raw input values should be scaled inside the backend before prediction
- The response should be consistent and easy for Streamlit to display

### Example API Request
```json
{
    "Time": 0.0,
    "V1": 0.0,
    "V2": 0.0,
    "...": 0.0,
    "Amount": 100.0
}

### Example API Response
```json
{
    "prediction": 1,
    "fraud_probability": 0.8732,
    "threshold": 0.5,
    "label": "Fraud"
}

# 9. Deployment Notes for Streamlit

The Streamlit frontend will serve as the user interface for entering transaction data and displaying fraud prediction results.

## Recommended Streamlit responsibilities

1. Provide input fields for transaction features
2. Send input data to the FastAPI `/predict` endpoint
3. Display:
   - fraud / not fraud result
   - fraud probability
   - selected threshold
   - optional explanation or SHAP plot in future version

## Recommended user flow

User enters transaction values → Streamlit sends request to FastAPI → FastAPI returns result → Streamlit displays prediction

## Suggested UI components

- numeric input fields
- submit button
- probability display
- fraud label display
- optional threshold slider
- optional explanation section

## Important notes

- Streamlit should not contain model logic directly if FastAPI is being used
- Streamlit should act as the presentation layer
- Prediction logic should remain centralized in the backend for consistency

# 10. Final Handover Summary

This notebook prepared the final fraud detection model for deployment.

The following tasks were completed:

1. The final trained model was loaded successfully
2. The preprocessing objects and feature columns were loaded
3. An end-to-end prediction function was created
4. The prediction pipeline was tested using sample input
5. Deployment artifacts were exported for backend use
6. Example input and output schemas were saved
7. Deployment notes for FastAPI and Streamlit were documented

The exported pipeline is now ready to be integrated into a real-time fraud detection application.

This notebook serves as the final technical handover from machine learning development to system deployment.